# MeshVTON — İnteraktif Test (kendi modelimiz)

Kişi fotoğrafı + giysi `.obj` (+ texture) yükle → eğitilmiş MeshVTON checkpoint'iyle try-on sonucu al.
İstersen 0/90/180/270° için ayrı ayrı üret (tek fotodan çok açı = 3D koşullama avantajı).

**Sıra:** hücreleri yukarıdan aşağı çalıştır. Ağır modeller (FLUX 12B, HMR2) bir kez yüklenir, sonraki üretimlerde tekrar yüklenmez.


In [ ]:
#@title 0) GitHub'dan son kodu çek (yeni push sonrası SADECE bunu çalıştır)
import os, sys
if not os.path.exists('/content/MeshVTON'):
    !git clone -q https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git fetch -q origin && git reset --hard origin/main   # lokal kalıntıları ez, son main'e eşitle
!git log --oneline -3

# Yüklü meshvton2 modüllerini boşalt -> runtime restart GEREKMEDEN yeni kod import edilir.
for m in [m for m in list(sys.modules) if m.startswith('meshvton2')]:
    del sys.modules[m]
if '/content/MeshVTON/v2' not in sys.path:
    sys.path.insert(0, '/content/MeshVTON/v2')
print('kod güncel. NOT: 3. hücredeki STATE eski sınıflardan kalmış olabilir —')
print('değişiklik model/builder koduna dokunduysa 3. hücreyi (ve 5-6yı) yeniden çalıştır.')


In [ ]:
#@title 1) Kurulum (~2-3 dk; Colab GPU şart — A100/L4 önerilir)
import os, importlib.util, subprocess, shutil
if not os.path.exists('/content/MeshVTON'):
    !git clone -q https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git pull -q

!pip -q install "diffusers>=0.34" "peft>=0.14" lpips einops sentencepiece trimesh smplx pyrender onnxruntime
!pip -q uninstall -y pyopengl PyOpenGL-accelerate > /dev/null 2>&1
!pip -q install "git+https://github.com/mmatl/pyopengl.git"
if importlib.util.find_spec('hmr2') is None:
    !pip -q install "git+https://github.com/shubham-goel/4D-Humans.git"
!apt-get -qq install -y libglu1-mesa libosmesa6 > /dev/null 2>&1

# EGL çalışıyorsa onu, yoksa OSMesa (yazılımsal)
_probe = subprocess.run(["python", "-c",
    'import os;os.environ["PYOPENGL_PLATFORM"]="egl";'
    'import pyrender;r=pyrender.OffscreenRenderer(16,16);r.delete();print("egl-ok")'],
    capture_output=True, text=True)
os.environ['PYOPENGL_PLATFORM'] = 'egl' if 'egl-ok' in _probe.stdout else 'osmesa'
print('GL platform:', os.environ['PYOPENGL_PLATFORM'])

# IDM-VTON yalnız KİŞİ ön-işlemesi için (parsing + openpose)
if not os.path.exists('/content/IDM-VTON'):
    !git clone -q https://github.com/yisol/IDM-VTON /content/IDM-VTON
from huggingface_hub import hf_hub_download
for repo_path in ('humanparsing/parsing_atr.onnx',
                  'humanparsing/parsing_lip.onnx',
                  'openpose/ckpts/body_pose_model.pth'):
    local = f'/content/IDM-VTON/ckpt/{repo_path}'
    if not (os.path.exists(local) and os.path.getsize(local) > 1_000_000):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(hf_hub_download('yisol/IDM-VTON', repo_path), local)
    assert os.path.getsize(local) > 1_000_000, f'bozuk indirme: {local}'

# FLUX.1 gated -> Colab Secrets'a HF_TOKEN ekle (soldaki anahtar simgesi)
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Kurulum OK')


In [ ]:
#@title 2) SMPL-X + HMR2 ağırlıkları (Drive'dan; build_conditioning bunlarsız çalışmaz)
import os, sys, glob, shutil
sys.path.insert(0, '/content/MeshVTON/v2')
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/MeshVTON'   # gerekirse kendi Drive düzenine göre değiştir

# SMPL-X neutral modeli
os.makedirs('/content/MeshVTON/checkpoints/pretrained/smplx', exist_ok=True)
!cp $D/smplx/SMPLX_NEUTRAL.* /content/MeshVTON/checkpoints/pretrained/smplx/
os.environ['SMPLX_MODEL_DIR'] = '/content/MeshVTON/checkpoints/pretrained/smplx'

# HMR2 ağırlıkları (bir kez ~3.5GB) + SMPL neutral pkl
from meshvton2.conditioning.body import _patch_torch_load_weights_only
import torch as _torch; _patch_torch_load_weights_only(_torch)
from hmr2.models import download_models
from hmr2.configs import CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)
smpl_dir = f"{CACHE_DIR_4DHUMANS}/data/smpl"; os.makedirs(smpl_dir, exist_ok=True)
cands = glob.glob(f'{D}/smpl/*neutral*lbs*.pkl') + glob.glob(f'{D}/smpl/SMPL_NEUTRAL.pkl')
assert cands, "SMPL neutral pkl yok -> Drive/MeshVTON/smpl/ (smplify.is.tue.mpg.de)"
shutil.copy(cands[0], f"{smpl_dir}/SMPL_NEUTRAL.pkl")
print('SMPL-X + HMR2 hazır')


In [ ]:
#@title 3) Modelleri BİR kez yükle (FLUX 12B + HMR2 + parser)  —  ~2-4 dk
# Ayarlar:
CHECKPOINT = '/content/drive/MyDrive/MeshVTON/v2_outputs/stage1/final.pt'  # eğitilmiş MeshVTON; None => stok FLUX (eğitimsiz)
SIZE = (1024, 768)  # (H, W) — base.yaml ile aynı, DEĞİŞTİRME

import yaml
from pathlib import Path
from meshvton2.conditioning.builder import assert_real_impl
from meshvton2.conditioning.body import build_hmr2_backend
from meshvton2.conditioning.person import PersonPreprocessor
from meshvton2.model.flux_tryon import FluxTryOnSampler

assert_real_impl()
base = yaml.safe_load(Path('/content/MeshVTON/v2/configs/base.yaml').read_text())

# globals cache: hücreyi tekrar çalıştırırsan yeniden yüklemez
if 'STATE' not in globals():
    STATE = {}
if 'prep' not in STATE:
    STATE['prep'] = PersonPreprocessor('/content/IDM-VTON')
if 'hmr2' not in STATE:
    STATE['hmr2'] = build_hmr2_backend()
if STATE.get('ckpt') != CHECKPOINT:
    STATE['sampler'] = FluxTryOnSampler(
        base['model']['flux_fill_repo'],
        checkpoint=CHECKPOINT, prompt=base['model']['prompt'])
    STATE['ckpt'] = CHECKPOINT
print('Modeller hazır. checkpoint =', CHECKPOINT)


In [ ]:
#@title 4) Kişi fotoğrafı + giysi .obj yükle (texture YOK)
# Kişi ve giysi AYRI klasörlere kaydedilir (aksi halde loader kişi fotosunu
# yanlışlıkla giysi texture'ı sanabiliyor). Texture kullanmıyoruz.
from google.colab import files
import os, shutil

PDIR, GDIR = '/content/uploads/person', '/content/uploads/garment'
for d in (PDIR, GDIR):
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)

print('>> KİŞİ fotoğrafını seç (jpg/png):')
up = files.upload()
PERSON_PATH = f'{PDIR}/' + next(iter(up))
open(PERSON_PATH, 'wb').write(next(iter(up.values())))

print('\n>> Giysi .OBJ dosyasını seç:')
up = files.upload()
OBJ_PATH = f'{GDIR}/' + next(iter(up))
open(OBJ_PATH, 'wb').write(next(iter(up.values())))

print('\nperson:', PERSON_PATH)
print('obj   :', OBJ_PATH)
print('NOT: texture yok -> model desen/rengi KOPYALAMAZ; yalnız yerleşim/şekil test edilir.')

# İPUCU: gerçek renk/desen için yükleme yerine DATASET giysisi kullan — loader
# .obj'nin yanındaki texture PNG'yi otomatik bulur (2. hücre zip'i açmıştı):
#   OBJ_PATH = '/content/MeshVTON/data/garments_3d/<giysi_klasörü>/mesh.obj'
#   !find /content/MeshVTON/data/garments_3d -name "*.obj" | head   # listelemek için


In [ ]:
#@title 5) Koşullama kur + TEŞHİS (GPU'da örnekleme YOK — önce hizayı gör)
ANGLES = [0]            #@param  ör. [0]  veya  [0, 90, 180, 270]  (0 = fotonun kendi açısı)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
from meshvton2.conditioning.builder import build_conditioning, PhotoView, OrbitView
from meshvton2.conditioning.garment import load_garment_asset
from meshvton2.conditioning.person import person_square_bbox
from meshvton2.conditioning.body import get_body_model
from meshvton2.conditioning import camera as cam_mod
from meshvton2.conditioning.render import render_body_mask

prep, hmr2 = STATE['prep'], STATE['hmr2']

# 1) kişi ön-işleme + poz/kamera — bbox artık parse'tan (kişi-merkezli kare):
#    tam-kare varsayılan, off-center kişide gövdeyi görüntü merkezine kaydırıyordu.
pp = prep.process(PERSON_PATH, size=SIZE)
params = hmr2(pp.image, bbox=person_square_bbox(pp))

# 2) giysi mesh — texture'sız yükle; yoksa düz gri referans (renk kopyalanmaz)
asset = load_garment_asset(OBJ_PATH, texture_path=None, allow_untextured=True,
                           garment_id=os.path.splitext(os.path.basename(OBJ_PATH))[0])
if asset.texture is None:
    tex = np.full((64, 64, 3), 200, np.uint8)
    uv = asset.uv if asset.uv is not None else np.zeros((len(asset.verts), 2), np.float32)
    asset = replace(asset, texture=tex, uv=uv)
    print('texture yok -> düz gri appearance referansı (renk/desen kopyalanmaz).')

# 3) koşullamalar (maske = parse ∪ giysi silüeti — hizalama düzeltmesi builder'da)
bundles = {a: build_conditioning(pp.image, params, asset,
                                 PhotoView() if a == 0 else OrbitView(a),
                                 size=SIZE, person_prep=pp)
           for a in ANGLES}

# ---- TEŞHİS (her zaman foto kamerasında: parse ile kıyas ancak 0°'de anlamlı) ----
b0 = bundles.get(0) or build_conditioning(pp.image, params, asset, PhotoView(),
                                          size=SIZE, person_prep=pp)
cam = b0.camera
body = get_body_model()(params)
h, w = SIZE

# a) gövde hizası: SMPL-X vertexleri fotoya projeksiyon
uv_pix = cam_mod.project(cam, body['verts'])
ok = ~np.isnan(uv_pix).any(1)
# b) gövde reprojection IoU: render silüeti vs parse kişi bölgesi
body_m = render_body_mask(body['verts'], body['faces'], cam, size=SIZE)
parse_full = cv2.resize(pp.parse, (w, h), interpolation=cv2.INTER_NEAREST) > 0
reproj_iou = (parse_full & body_m).sum() / max((parse_full | body_m).sum(), 1)
# c) maske (kırmızı) vs giysi silüeti (yeşil) kaplaması
sil = b0.control_depth_sil[2].numpy() > 0
msk = b0.inpaint_mask[0].numpy() > 0.5
cover = (sil & msk).sum() / max(sil.sum(), 1)   # silüetin ne kadarı maske içinde

overlay = pp.image.copy()
overlay[msk] = (overlay[msk] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
overlay[sil] = (overlay[sil] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
fig, ax = plt.subplots(1, 3, figsize=(13, 6))
ax[0].imshow(pp.image); ax[0].scatter(uv_pix[ok, 0][::25], uv_pix[ok, 1][::25], s=0.3, c='cyan')
ax[0].set_title(f'gövde projeksiyonu (IoU={reproj_iou:.2f})')
ax[1].imshow(overlay); ax[1].set_title('maske(kırmızı) vs silüet(yeşil)')
ax[2].imshow(body_m, cmap='gray'); ax[2].set_title('render gövde silüeti')
for x in ax: x.axis('off')
plt.tight_layout(); plt.show()

meta = b0.meta
print(f"reprojection IoU      : {reproj_iou:.3f}  (kapı: >= 0.70)")
print(f"silüet-maske kapsaması: {cover:.3f}  (>= ~0.95 beklenir — birleşim düzeltmesiyle)")
print(f"drape_extent_ratio    : {meta.get('drape_extent_ratio'):.3f}  (üst giysi/tüm-vücut köşegeni: ~0.25-0.6 normal)")
print(f"clearance_ratio       : {meta.get('clearance_ratio'):.3f}   penetrasyon: {meta.get('penetration_depth', 0)*1000:.1f} mm")
if reproj_iou < 0.70:
    print('UYARI: kamera/SMPL fit zayıf — gövde kişiyle hizalı değil; sonuç kayık olur.')
_er = meta.get('drape_extent_ratio') or 0.4
if _er > 1.5 or _er < 0.15:
    print('UYARI: drape patlamış/çökmüş görünüyor — mesh ölçeği/ekseni şüpheli (CLOTH3D metrik + Z-up olmalı).')


In [ ]:
#@title 6) Try-on üret + göster/kaydet
STEPS = 28              #@param
GUIDANCE = 1.0          #@param  1.0 = eğitim değeri; 3.5-30 arası renk doygunluğunu artırabilir (solgunluk çaresi)
SEED = 0               #@param

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from meshvton2.utils.image_utils import tensor_to_pil

sampler = STATE['sampler']
os.makedirs('/content/results', exist_ok=True)
results = {}
for a, bundle in bundles.items():
    pred = sampler.sample(bundle, steps=STEPS, guidance=GUIDANCE, seed=SEED)
    results[a] = pred
    out = f'/content/results/{asset.garment_id}_a{a}.png'
    Image.fromarray(pred).save(out)
    print('kaydedildi:', out)

appref = np.asarray(tensor_to_pil(next(iter(bundles.values())).appearance_ref))
cols = 2 + len(results)
fig, ax = plt.subplots(1, cols, figsize=(4 * cols, 6))
ax[0].imshow(pp.image);   ax[0].set_title('kişi (girdi)')
ax[1].imshow(appref);     ax[1].set_title('giysi referansı')
for j, a in enumerate(results):
    ax[2 + j].imshow(results[a]); ax[2 + j].set_title(f'MeshVTON {a}°')
for x in ax: x.axis('off')
plt.tight_layout(); plt.show()
